In [ ]:
import pandas as pd
import re
import ast  # For safely evaluating strings as lists

# 1. Load Data
df = pd.read_parquet("hf://datasets/AWeirdDev/all-recipes-xs/data/train-00000-of-00001.parquet")

# 2. Initial Inspection (CRITICAL - Do this yourself!)
print(df.head())
print(df.info())
#for col in df.columns:
#    print(f"Column: {col}")
#    print(df[col].unique())
#    print("-" * 20)  # This will give you the raw values, which are essential

# 3. Start with a Subset
subset_df = df.head(5000)  # Adjust subset size as needed

# 4. Cleaning Functions (Iterative)

# A. Clean Ingredients (Most Important First)
def clean_ingredients(ingredients_str): # assuming ingredients are stored as strings representing lists
    try:
        ingredients_list = ast.literal_eval(ingredients_str) #safely convert string to list
    except (SyntaxError, ValueError): #if the string is not a list, return an empty list
        return []
    cleaned_list = []
    for ingredient in ingredients_list:
        ingredient = ingredient.lower().strip() #convert to lower and remove leading/trailing spaces
        ingredient = re.sub(r"^[0-9]+ (?:cup|tablespoon|teaspoon|gram|pound|ounce|...)? ", "", ingredient).strip() #remove leading number and unit
        ingredient = re.sub(r"\(.*?\)", "", ingredient).strip() #remove anything in parenthesis
        # Add more regex cleaning here as needed based on what you see in df[‘ingredients’].unique()
        cleaned_list.append(ingredient)
    return cleaned_list

subset_df['ingredients'] = subset_df['ingredients'].apply(clean_ingredients)
print("Cleaned Ingredients Example:") #to see if it works
print(subset_df['ingredients'].head())

# B. Clean Instructions
def clean_instructions(instructions_str): #assuming instructions are stored as strings
    try:
        instructions_list = ast.literal_eval(instructions_str) #safely convert string to list
    except (SyntaxError, ValueError):
        return []
    cleaned_list = []
    for instruction in instructions_list:
        instruction = re.sub(r"<.*?>", "", instruction)  # Remove HTML tags
        instruction = instruction.replace("\n", " ").replace("\r", " ").strip() #remove new lines and carriage returns
        # Add other text cleaning steps (e.g., handling special characters)
        cleaned_list.append(instruction)
    return cleaned_list

subset_df['steps'] = subset_df['steps'].apply(clean_instructions)

print("\nCleaned Instructions Example:")
print(subset_df['steps'].head())

# C. Clean Titles (Example)
def clean_title(title):
    title = title.lower().strip()
    # Add more title cleaning as needed (e.g., remove extra words)
    return title

# ----> Apply clean_title to the 'name' column instead of 'title'
subset_df['name'] = subset_df['name'].apply(clean_title)

print("\nCleaned Title Example:")
print(subset_df['name'].head()) # ----> Print the cleaned names

# ... (Add more cleaning functions for other columns as needed)

# 5. Evaluation (Crucial Step)
# Examine the cleaned subset to see if the cleaning worked as expected.
# If you have a model, test its performance on the cleaned subset.

# 6. Apply to Full Dataset (After thorough evaluation)
df['ingredients'] = df['ingredients'].apply(clean_ingredients)
df['steps'] = df['steps'].apply(clean_instructions)
# ----> Apply clean_title to the 'name' column in the full dataset
df['name'] = df['name'].apply(clean_title)

# ... (Apply other cleaning functions to the full dataset)

# 7. Save Cleaned Data (Important)
df.to_parquet("cleaned_recipes.parquet") #save as parquet for efficiency

print("Cleaned data saved to cleaned_recipes.parquet")

                             name  \
0  Dutch Oven Crunchy Corned Beef   
1      Spicy Spicy Ranch Dressing   
2   Amazing Oven-Roasted Potatoes   
3                Easy Curry Sauce   
4              Lemon Ginger Water   

                                              review  rating  \
0  I found this recipe attached to a 6-pack of be...     5.0   
1  This spicy ranch dip with hot sauce and cayenn...     4.5   
2  These oven-roasted potatoes are golden on the ...     4.8   
3  This curry sauce is versatile and can be serve...     4.1   
4  This lightly sweetened lemon ginger water not ...     5.0   

                                                meta  \
0  {'active_time': None, 'additional_time': None,...   
1  {'active_time': None, 'additional_time': '1 hr...   
2  {'active_time': None, 'additional_time': None,...   
3  {'active_time': None, 'additional_time': None,...   
4  {'active_time': None, 'additional_time': None,...   

                                         ingredients  \

In [ ]:
print(subset_df.columns)

Index(['name', 'review', 'rating', 'meta', 'ingredients', 'steps',
       'cooks_note', 'editors_note', 'nutrition_facts', 'url'],
      dtype='object')
